# AIPerf Use Case 1 — Simple Synthetic Profiling (+ Concurrency Sweep / Pareto)

Profile an OpenAI-compatible endpoint with fully **synthetic traffic** (fixed input/output sequence lengths), then extend the same command into a **concurrency sweep** to draw the cost-vs-UX Pareto curve.

Based on the [AIPerf office-hours gist](https://gist.github.com/BenHamm/31c648f7d7331c94c1f3a45859db6677) — see `docs/reference/aiperf-office-hours.md` for the distilled notes. These use-case notebooks demonstrate AIPerf capabilities directly; the repo's per-scenario bash scripts remain the source of truth for the Model Selection / Sizing suites.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — synthetic prompts](#3-input--synthetic-prompts)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)
6. [Concurrency sweep and Pareto curve](#6-concurrency-sweep-and-pareto-curve)

## 1. Overview

`aiperf profile` simulates N concurrent users (`--concurrency`) sending a total of `--request-count` requests, each with a synthetic prompt of `--synthetic-input-tokens-mean` tokens (ISL) and a forced generation length of `--output-tokens-mean` tokens (OSL). With `--streaming` enabled, AIPerf measures:

- **TTFT** (time to first token) — responsiveness before generation starts.
- **ITL** (inter-token latency) — streaming smoothness.
- **Request latency**, **output token throughput** (system-wide), **request throughput**.

Two details that are easy to get wrong:

- `--streaming` is what makes TTFT/ITL measurable at all.
- `--tokenizer` must match the served model, or token counts (and therefore every tokens/sec number) are wrong.

Reasoning models: AIPerf separates reasoning tokens from visible output tokens by default; thinking tokens count toward OSL.

## 2. Requirements

- `aiperf` CLI — the setup cell below installs it via pip if it isn't already on `PATH`. The office-hours gist pinned `release/0.3.0`; pin whatever version you use (repo convention: record the AIPerf version per run).
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- A Hugging Face token (`HF_TOKEN` in the config cell) if the tokenizer repo is gated (e.g. Llama, Mistral).
- Tip: AIPerf's live dashboard is designed for a terminal. If it renders poorly inside Jupyter, check `aiperf profile --help` for a plain/simple UI option in your version.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

if shutil.which("aiperf") is None:
    print("aiperf CLI not found — installing into this kernel's environment ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "aiperf"], check=True)

aiperf_path = shutil.which("aiperf")
assert aiperf_path, "aiperf still not found after install — restart the kernel and rerun this cell."
print(f"aiperf found at: {aiperf_path}")
version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
print((version.stdout or version.stderr).strip())


## 3. Input — synthetic prompts

There is no prompt file in this use case: AIPerf generates random token sequences of the requested length. That makes runs perfectly repeatable and comparable, at the cost of realism — no natural language, no shared prefixes, so KV-cache/prefix-reuse effects are invisible (see the Use Case 3 notebook for trace replay, which fixes that).

| Flag | Meaning |
|---|---|
| `--synthetic-input-tokens-mean` | ISL — prompt length in tokens (gist shorthand: `--isl`) |
| `--synthetic-input-tokens-stddev` | Sample ISL from a distribution instead of a fixed value |
| `--output-tokens-mean` | OSL — forced generation length (gist shorthand: `--osl`) |
| `--output-tokens-stddev` | Sample OSL from a distribution |
| `--random-seed` | Make the synthetic generation reproducible |

The gist's demo: ISL 1000 / OSL 500 ≈ "600 words in, 300 words back", i.e. a plausible chat-style exchange.

## 4. Test run

Set `MODEL` and `URL`, then run. Defaults below are scaled down from the gist (concurrency 100 / 1000 requests against 8× H200) so the first run finishes quickly against a modest endpoint — scale them back up as your deployment allows.

In [ ]:
# ---- Required ----------------------------------------------------------
MODEL = ""       # e.g. "qwen3-0.6b" — the model name the endpoint serves
URL = ""         # e.g. "http://localhost:8000"

# ---- Workload ----------------------------------------------------------
CONCURRENCY = 10          # gist demo used 100
REQUEST_COUNT = 100       # gist demo used 1000
ISL = 1000                # input sequence length (tokens)
OSL = 500                 # output sequence length (tokens)
TOKENIZER = ""            # HF repo id or local dir; empty = use MODEL as tokenizer name
RANDOM_SEED = "42"

# ---- Hugging Face ------------------------------------------------------
# aiperf downloads the tokenizer from Hugging Face. A token is only needed
# for gated repos (e.g. Llama, Mistral) or private tokenizers.
HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN  # inherited by the aiperf subprocess

OUTPUT_DIR = "artifacts/aiperf-uc1-synthetic"


In [ ]:
assert MODEL and URL, "Set MODEL and URL in the config cell above."

# The command, written exactly as you would type it in a terminal.
# .split() turns it into the argument list subprocess expects.
cmd = f"""
aiperf profile
    --model {MODEL}
    --url {URL}
    --endpoint-type chat
    --streaming
    --concurrency {CONCURRENCY}
    --request-count {REQUEST_COUNT}
    --synthetic-input-tokens-mean {ISL}
    --output-tokens-mean {OSL}
    --random-seed {RANDOM_SEED}
    --tokenizer {TOKENIZER or MODEL}
    --artifact-dir {OUTPUT_DIR}
""".split()

print("$ " + " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
print(f"\nexit code: {result.returncode}")

# NOTE: --synthetic-input-tokens-mean / --output-tokens-mean are the canonical
# long names for the gist's --isl / --osl shorthands. Used here because they are
# guaranteed across AIPerf versions and match this repo's scenario scripts
# (e.g. --output-tokens-mean in model-selection/scripts/run_content_generation.sh).


## 5. Results analysis

AIPerf writes three exports to the artifact directory:

- `profile_export_aiperf.csv` — aggregated statistics (avg/min/max/p50/p90/p99/std per metric).
- `profile_export_aiperf.json` — the same statistics plus the run's full configuration.
- `profile_export.jsonl` — one record **per request** (analyzed in depth in the Use Case 2 notebook).

Reference points from the gist's (overprovisioned) demo at concurrency 100: TTFT avg ≈ 347 ms, request latency avg ≈ 2.1 s, ITL avg ≈ 3.6 ms, system throughput ≈ 22.5K tokens/sec.

In [ ]:
import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
csv_path = next(artifact_dir.rglob("profile_export_aiperf.csv"), None)
assert csv_path is not None, f"No profile_export_aiperf.csv under {artifact_dir} — did the Test run complete?"

stats = pd.read_csv(csv_path)
metric_col = stats.columns[0]
pd.set_option("display.max_rows", None)
stats


In [ ]:
key_metric_patterns = [
    "time to first token",
    "inter token latency",
    "request latency",
    "output token throughput",
    "request throughput",
]
mask = stats[metric_col].str.lower().apply(lambda n: any(p in n for p in key_metric_patterns))
stats[mask]


## 6. Concurrency sweep and Pareto curve

Rerunning the identical benchmark at several concurrency levels shows the fundamental trade-off:

- **TPS per GPU** (system throughput ÷ GPU count) — resource/cost efficiency, improves with concurrency up to a saturation point.
- **TPS per user** (throughput ÷ concurrency) — individual user experience, degrades monotonically with concurrency.

In the gist's demo, per-GPU efficiency peaked at concurrency 200 (~18K TPS/GPU) and *regressed* at 500 (suspected client-side network/port-exhaustion effects), while TTFT grew from ~250 ms to ~1.1 s. The curve is how you pick an operating point and set a per-replica load-balancer cap. This is the same idea as the repo's `sizing/` concurrency ladder (1/5/10/25/50/100/200) — use that suite for the real sizing workflow.

The sweep below is **off by default** (`RUN_SWEEP = False`) because it reruns the full benchmark once per level.

In [ ]:
RUN_SWEEP = False
SWEEP_CONCURRENCIES = [1, 5, 10, 25]   # gist used [10, 50, 100, 200, 500]
NUM_GPUS = 1                           # GPUs behind the endpoint, for the TPS/GPU axis

if RUN_SWEEP:
    for c in SWEEP_CONCURRENCIES:
        sweep_cmd = list(cmd)
        sweep_cmd[sweep_cmd.index("--concurrency") + 1] = str(c)
        sweep_cmd[sweep_cmd.index("--artifact-dir") + 1] = f"{OUTPUT_DIR}-sweep-c{c}"
        print("$ " + " ".join(sweep_cmd))
        subprocess.run(sweep_cmd, cwd=REPO_ROOT, text=True)
else:
    print("RUN_SWEEP is False — flip it to True to run the sweep.")


In [ ]:
import matplotlib.pyplot as plt


def stat_value(df, name_contains, col="avg"):
    m = df[df.iloc[:, 0].str.lower().str.contains(name_contains)]
    return float(m[col].iloc[0]) if not m.empty and col in df.columns else None


rows = []
for c in SWEEP_CONCURRENCIES:
    d = REPO_ROOT / f"{OUTPUT_DIR}-sweep-c{c}"
    p = next(d.rglob("profile_export_aiperf.csv"), None)
    if p is None:
        continue
    df = pd.read_csv(p)
    total_tps = stat_value(df, "output token throughput")
    rows.append({
        "concurrency": c,
        "total_tps": total_tps,
        "tps_per_gpu": total_tps / NUM_GPUS if total_tps else None,
        "tps_per_user": total_tps / c if total_tps else None,
        "ttft_avg_ms": stat_value(df, "time to first token"),
    })

sweep = pd.DataFrame(rows)
if sweep.empty:
    print("No sweep artifacts found — run the sweep cell first.")
else:
    display(sweep)
    ax = sweep.plot(x="tps_per_user", y="tps_per_gpu", marker="o", legend=False, figsize=(7, 4.5))
    for _, r in sweep.iterrows():
        ax.annotate(f"c={int(r.concurrency)}", (r.tps_per_user, r.tps_per_gpu),
                    textcoords="offset points", xytext=(6, 4))
    ax.set_xlabel("TPS per user (user experience →)")
    ax.set_ylabel("TPS per GPU (cost efficiency →)")
    ax.set_title("Pareto: cost efficiency vs. user experience")
    plt.tight_layout()


### Notes

- Each sweep run writes to its own `-sweep-c<N>` artifact directory so exports never overwrite each other.
- You cannot optimize both axes at once: pick the point that meets your per-user SLO at the highest per-GPU efficiency, then cap each replica at that concurrency.
- A/B testing (vLLM vs. SGLang vs. TRT-LLM, or config changes): keep this exact command fixed and only change the endpoint — identical workload is what makes the numbers comparable.